# small-vs-frontier: Colab GPU steps

Every GPU step of the project, in order. Claude Code drives this notebook through colab-mcp; the same cells can be run by hand. Runtime: **T4 GPU** (Runtime → Change runtime type).

## Phase 1: setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || echo "NO GPU"

Mounting Drive needs you to approve a Google popup, so run this cell by hand.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/small-vs-frontier")
for sub in ["checkpoints", "adapters", "results", "gguf"]:
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)
print(sorted(p.name for p in DRIVE_ROOT.iterdir()))

Add `HF_TOKEN` in the Secrets panel (🔑, notebook access on). Secrets can only be read when the cell is run **from the Colab UI**, so run the next cell by hand after every runtime restart. Processes started from the notebook inherit it.

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # never printed
print("HF_TOKEN set:", bool(os.environ["HF_TOKEN"]))

Pinned to the versions verified on a T4 (see `requirements-colab.txt`).

In [ ]:
%%capture pip_log
!pip install -q unsloth==2026.10.5 transformers==5.17.0 trl==1.15.0 peft==0.21.1 bitsandbytes==0.50.2 accelerate==1.15.0 datasets==4.8.5 pyyaml scikit-learn python-dotenv

In [ ]:
print("\n".join(pip_log.stdout.splitlines()[-15:]))
print(pip_log.stderr[-2000:])

Import check in a fresh process, so modules already loaded in the kernel can't hide a version clash.

In [ ]:
%%bash
python - <<'PYEOF' 2>&1 | grep -v -i "warn" | tail -25
import platform
import unsloth  # must be imported before transformers/trl/peft
import torch, transformers, trl, peft, bitsandbytes, datasets, accelerate
print("python", platform.python_version())
for m in (unsloth, torch, transformers, trl, peft, bitsandbytes, datasets, accelerate):
    print(f"{m.__name__}=={m.__version__}")
print("cuda available:", torch.cuda.is_available(), "| cuda", torch.version.cuda, "|", torch.cuda.get_device_name(0))
x = torch.randn(1024, 1024, device="cuda", dtype=torch.float16)
print("fp16 matmul ok:", (x @ x).isfinite().all().item())
PYEOF

## Phase 2: code and data

Project code lives on Drive (`small-vs-frontier/code`) so it survives runtime resets. Until the repo is pushed, Claude Code copies `src/`, `scripts/` and `configs/` there and checks each file's sha256 against the laptop copy.

In [ ]:
# Project code lives on Drive so it survives runtime resets.
# Once the repo is on GitHub this becomes: !git clone https://github.com/Ham12-3/fine-tune {CODE}
# Until then Claude Code copies src/, scripts/ and configs/ here and checks each file's sha256 against the laptop.
import hashlib
CODE = DRIVE_ROOT / "code"
for p in sorted(CODE.rglob("*")):
    if p.is_file() and "data" not in p.relative_to(CODE).parts[:1]:
        print(hashlib.sha256(p.read_bytes()).hexdigest()[:16], p.relative_to(CODE))

Rebuild the splits (downloads the pinned CSVs and checks their sha256). The hashes must match `data/processed/manifest.json` on the laptop.

In [ ]:
!cd {CODE} && PYTHONPATH=src python scripts/prepare_data.py
import json
m = json.loads((CODE / "data/processed/manifest.json").read_text())
print(m["labels_sha256"][:16], {k: v["sha256"][:16] for k, v in m["files"].items()})